# HDF Output Options Read Benchmark

Use the `BaldEagleCrkMulti2D` example project and plan `06` to compare HDF write settings by measuring ras-commander HDF read performance after the plans have been computed.

The compute phase only populates the HDF files. The benchmark phase measures calls into `HdfMesh` and `HdfResultsMesh`, including mesh geometry reads, summary-output reads, and full time-series reads.

## Workflow

1. Extract a working copy of `BaldEagleCrkMult2D`.
2. Clone template plan `06` once for each HDF settings permutation.
3. Apply each HDF output profile or explicit option set to its clone.
4. Compute missing clone results if requested.
5. Benchmark HDF reads from the populated `.p##.hdf` files.

For storage topology testing, copy or extract the same working project to HDD, SSD, and NAS-backed folders, then rerun the benchmark cells against each folder.

In [1]:
from pathlib import Path
from time import perf_counter
import gc
import json

import h5py
import pandas as pd

from ras_commander import (
    init_ras_project,
    RasCmdr,
    RasExamples,
    RasPlan,
    HdfMesh,
    HdfResultsMesh,
)

In [2]:
PROJECT_NAME = "BaldEagleCrkMulti2D"
TEMPLATE_PLAN = "06"
RAS_VERSION = "7.0"

WORKING_ROOT = Path.cwd() / "working" / "hdf_output_options_benchmark"
PROJECT_FOLDER = WORKING_ROOT / PROJECT_NAME

# Set True to delete and re-extract the working example project.
RESET_EXTRACTED_PROJECT = False

# Set True once to compute missing clone HDFs. Leave False when only re-running read benchmarks.
COMPUTE_MISSING_RESULTS = False
FORCE_RECOMPUTE = False
NUM_CORES = 4

# Extraction timing controls. The first repetition is usually affected by OS cache state.
READ_REPETITIONS = 3

# Keep this empty for pure HDF write-parameter tests. Add variables only when benchmarking extra outputs.
DEFAULT_ADDITIONAL_OUTPUT_VARIABLES = []
# Example:
# DEFAULT_ADDITIONAL_OUTPUT_VARIABLES = ["Face Flow", "Face Shear Stress", "Cell Velocity"]

In [3]:
PERMUTATIONS = [
    {
        "label": "baseline_p06_clone",
        "shortid": "HDF_base",
        "title": "HDF bench baseline",
        "profile": None,
        "options": {},
        "additional_variables": [],
    },
    {
        "label": "speed_profile",
        "shortid": "HDF_speed",
        "title": "HDF bench speed",
        "profile": "speed",
        "options": {},
        "additional_variables": DEFAULT_ADDITIONAL_OUTPUT_VARIABLES,
    },
    {
        "label": "balanced_profile",
        "shortid": "HDF_bal",
        "title": "HDF bench balanced",
        "profile": "balanced",
        "options": {},
        "additional_variables": DEFAULT_ADDITIONAL_OUTPUT_VARIABLES,
    },
    {
        "label": "size_profile",
        "shortid": "HDF_size",
        "title": "HDF bench size",
        "profile": "size",
        "options": {},
        "additional_variables": DEFAULT_ADDITIONAL_OUTPUT_VARIABLES,
    },
    {
        "label": "nas_profile",
        "shortid": "HDF_nas",
        "title": "HDF bench NAS",
        "profile": "nas",
        "options": {},
        "additional_variables": DEFAULT_ADDITIONAL_OUTPUT_VARIABLES,
    },
    # Add explicit permutations as needed. Keep shortid <= 24 chars and title <= 32 chars.
    # {
    #     "label": "compression_9_4mb",
    #     "shortid": "HDF_c9_4mb",
    #     "title": "HDF bench c9 4mb",
    #     "profile": None,
    #     "options": {
    #         "write_warmup": True,
    #         "write_time_slices": False,
    #         "hdf_flush": False,
    #         "compression": 9,
    #         "chunk_size_mb": 4,
    #         "spatial_parts": 1,
    #         "use_max_rows": True,
    #         "fixed_rows": 1,
    #     },
    #     "additional_variables": DEFAULT_ADDITIONAL_OUTPUT_VARIABLES,
    # },
]

pd.DataFrame(PERMUTATIONS)[["label", "shortid", "profile", "options", "additional_variables"]]

,label,shortid,profile,options,additional_variables
0,baseline_p06_clone,HDF_base,NaN,{},[]
1,speed_profile,HDF_speed,speed,{},[]
2,balanced_profile,HDF_bal,balanced,{},[]
3,size_profile,HDF_size,size,{},[]
4,nas_profile,HDF_nas,nas,{},[]


In [4]:
WORKING_ROOT.mkdir(parents=True, exist_ok=True)

if RESET_EXTRACTED_PROJECT or not PROJECT_FOLDER.exists():
    project_folder = RasExamples.extract_project(PROJECT_NAME, output_path=WORKING_ROOT)
else:
    project_folder = PROJECT_FOLDER

ras = init_ras_project(project_folder, RAS_VERSION, load_results_summary=False)
ras.plan_df[["plan_number", "Plan Title", "Short Identifier", "HDF_Results_Path", "full_path"]]

2026-04-24 13:37:57 - ras_commander.RasExamples - INFO - Found zip file: C:\Users\bill\AppData\Local\ras-commander\examples\Example_Projects_7_0.zip


2026-04-24 13:37:57 - ras_commander.RasExamples - INFO - Loading project data from CSV...


2026-04-24 13:37:57 - ras_commander.RasExamples - INFO - Loaded 67 projects from CSV.


2026-04-24 13:37:57 - ras_commander.RasExamples - INFO - ----- RasExamples Extracting Project -----


2026-04-24 13:37:57 - ras_commander.RasExamples - INFO - Extracting project 'BaldEagleCrkMulti2D'


...

2026-04-24 13:37:59 - ras_commander.hdf.HdfBase - INFO - Converted WKT to EPSG:2271 from HDF file BaldEagleDamBrk.g09.hdf


2026-04-24 13:37:59 - ras_commander.RasPrj - INFO - ras-commander v0.95.0 | An open-source project of CLB Engineering Corporation (https://clbengineering.com/) | Docs: https://ras-commander.readthedocs.io | GitHub: https://github.com/gpt-cmdr/ras-commander


2026-04-24 13:37:59 - ras_commander.RasPrj - INFO - Project initialized: BaldEagleDamBrk | Folder: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D


2026-04-24 13:37:59 - ras_commander.RasPrj - INFO - Using HEC-RAS executable: C:\Program Files (x86)\HEC\HEC-RAS\7.0\Ras.exe


,plan_number,Plan Title,Short Identifier,HDF_Results_Path,full_path
0,13,PMF with Multi 2D Areas,PMF Multi 2D,None,G:\GH\ras-commander\working\hdf_output_options...
1,15,1d-2D Dambreak Refined Grid,1D-2D Refined Grid,None,G:\GH\ras-commander\working\hdf_output_options...
2,17,2D to 1D No Dam,2D to 1D No Dam,None,G:\GH\ras-commander\working\hdf_output_options...
3,18,2D to 2D Run,2D to 2D Run,None,G:\GH\ras-commander\working\hdf_output_options...
4,19,SA to 2D Dam Break Run,SA to 2D Dam Break,None,G:\GH\ras-commander\working\hdf_output_options...
5,03,Single 2D Area - Internal Dam Structure,Single 2D,None,G:\GH\ras-commander\working\hdf_output_options...
6,04,SA to 2D Area Conn - 2D Levee Structure,2D Levee Struc,None,G:\GH\ras-commander\working\hdf_output_options...
7,02,SA to Detailed 2D Breach,SA-2D Det Brch,None,G:\GH\ras-commander\working\hdf_output_options...
8,01,SA to Detailed 2D Breach FEQ,SA-2D Det FEQ,None,G:\GH\ras-commander\working\hdf_output_options...
9,05,Single 2D area with Bridges FEQ,Single 2D Bridges FEQ,None,G:\GH\ras-commander\working\hdf_output_options...


In [5]:
if TEMPLATE_PLAN not in set(ras.plan_df["plan_number"].astype(str)):
    raise ValueError(
        f"Template plan {TEMPLATE_PLAN} not found. Available plans: "
        f"{ras.plan_df['plan_number'].tolist()}"
    )

print("Available HDF profiles:")
print(json.dumps(RasPlan.list_hdf_output_setting_profiles(), indent=2))

print(f"\nKnown additional HDF output variables: {len(RasPlan.list_available_hdf_output_variables())}")
RasPlan.list_available_hdf_output_variables()[:10]

Available HDF profiles:
{
  "balanced": {
    "write_warmup": true,
    "write_time_slices": false,
    "hdf_flush": false,
    "compression": 4,
    "chunk_size_mb": 4,
    "spatial_parts": 1,
    "use_max_rows": true,
    "fixed_rows": 1
  },
  "speed": {
    "write_warmup": true,
    "write_time_slices": false,
    "hdf_flush": false,
    "compression": 1,
    "chunk_size_mb": 4,
    "spatial_parts": 1,
    "use_max_rows": true,
    "fixed_rows": 1
  },
  "size": {
    "write_warmup": true,
    "write_time_slices": false,
    "hdf_flush": false,
    "compression": 6,
    "chunk_size_mb": 4,
    "spatial_parts": 1,
    "use_max_rows": true,
    "fixed_rows": 1
  },
  "nas": {
    "write_warmup": true,
    "write_time_slices": false,
    "hdf_flush": false,
    "compression": 6,
    "chunk_size_mb": 2,
    "spatial_parts": 1,
    "use_max_rows": true,
    "fixed_rows": 1
  }
}

Known additional HDF output variables: 44


['Cell Cumulative Excess Depth',
 'Cell Cumulative Infiltration Depth',
 'Cell Cumulative Percolation Depth',
 'Cell Cumulative Precipitation Depth',
 'Cell Eddy Viscosity',
 'Cell Cumulative Evapotranspiration Depth',
 'Cell Evapotranspiration Potential Rate',
 'Cell Evapotranspiration Rate',
 'Cell Excess Rate',
 'Cell Flow Balance (inflows - outflows)']

In [6]:
def refresh_project():
    global ras
    ras = init_ras_project(project_folder, RAS_VERSION, load_results_summary=False)
    return ras


def find_plan_by_shortid(shortid: str):
    ras.plan_df = ras.get_plan_entries()
    shortids = ras.plan_df["Short Identifier"].fillna("").astype(str).str.strip()
    match = ras.plan_df[shortids == shortid[:24]]
    if match.empty:
        return None
    return str(match.iloc[0]["plan_number"])


def expected_plan_hdf_path(plan_number: str) -> Path:
    return Path(ras.project_folder) / f"{ras.project_name}.p{str(plan_number).zfill(2)}.hdf"


def ensure_benchmark_plan(perm: dict) -> dict:
    plan_number = find_plan_by_shortid(perm["shortid"])
    created = False

    if plan_number is None:
        plan_number = RasPlan.clone_plan(
            TEMPLATE_PLAN,
            new_plan_shortid=perm["shortid"],
            new_title=perm["title"],
            description=(
                f"HDF read benchmark clone of plan {TEMPLATE_PLAN}. "
                f"Permutation: {perm['label']}"
            ),
            ras_object=ras,
        )
        created = True
        refresh_project()

    if perm.get("profile"):
        RasPlan.apply_hdf_output_profile(
            plan_number,
            profile=perm["profile"],
            additional_variables=perm.get("additional_variables") or [],
            ras_object=ras,
        )
    elif perm.get("options"):
        RasPlan.set_hdf_output_options(
            plan_number,
            ras_object=ras,
            **perm["options"],
        )
        if perm.get("additional_variables"):
            RasPlan.set_hdf_output_variables(
                plan_number,
                perm["additional_variables"],
                enabled=True,
                ras_object=ras,
            )
    elif perm.get("additional_variables"):
        RasPlan.set_hdf_output_variables(
            plan_number,
            perm["additional_variables"],
            enabled=True,
            ras_object=ras,
        )

    refresh_project()
    hdf_path = expected_plan_hdf_path(plan_number)
    return {
        "label": perm["label"],
        "shortid": perm["shortid"],
        "profile": perm.get("profile"),
        "plan_number": plan_number,
        "created": created,
        "hdf_path": hdf_path,
        "hdf_exists": hdf_path.exists(),
        "options": RasPlan.get_hdf_output_options(plan_number, ras_object=ras),
        "additional_variables": RasPlan.get_hdf_output_variables(plan_number, ras_object=ras),
    }


benchmark_plans = [ensure_benchmark_plan(perm) for perm in PERMUTATIONS]
plans_df = pd.DataFrame(benchmark_plans)
plans_df[["label", "plan_number", "profile", "hdf_exists", "hdf_path", "options", "additional_variables"]]

2026-04-24 13:37:59 - ras_commander.RasUtils - INFO - File cloned from G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p06 to G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07


2026-04-24 13:37:59 - ras_commander.RasUtils - INFO - Successfully updated file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07


2026-04-24 13:37:59 - ras_commander.RasUtils - INFO - Project file updated with new Plan entry: 07


2026-04-24 13:37:59 - ras_commander.RasMap - INFO - Successfully parsed RASMapper file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.rasmap


2026-04-24 13:37:59 - ras_commander.hdf.HdfBase - INFO - Found projection in HDF file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.g09.hdf


...

2026-04-24 13:38:04 - ras_commander.RasPrj - INFO - ras-commander v0.95.0 | An open-source project of CLB Engineering Corporation (https://clbengineering.com/) | Docs: https://ras-commander.readthedocs.io | GitHub: https://github.com/gpt-cmdr/ras-commander


2026-04-24 13:38:04 - ras_commander.RasPrj - INFO - Project initialized: BaldEagleDamBrk | Folder: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D


2026-04-24 13:38:04 - ras_commander.RasPrj - INFO - Using HEC-RAS executable: C:\Program Files (x86)\HEC\HEC-RAS\7.0\Ras.exe


2026-04-24 13:38:04 - ras_commander.RasPlan - INFO - Found 9 HDF output variables in plan


,label,plan_number,profile,hdf_exists,hdf_path,options,additional_variables
0,baseline_p06_clone,07,NaN,False,G:\GH\ras-commander\working\hdf_output_options...,"{'write_warmup': False, 'write_time_slices': F...","[Cell Invert Depth, Cell Velocity, Cell Flow B..."
1,speed_profile,08,speed,False,G:\GH\ras-commander\working\hdf_output_options...,"{'write_warmup': True, 'write_time_slices': Fa...","[Cell Invert Depth, Cell Velocity, Cell Flow B..."
2,balanced_profile,09,balanced,False,G:\GH\ras-commander\working\hdf_output_options...,"{'write_warmup': True, 'write_time_slices': Fa...","[Cell Invert Depth, Cell Velocity, Cell Flow B..."
3,size_profile,10,size,False,G:\GH\ras-commander\working\hdf_output_options...,"{'write_warmup': True, 'write_time_slices': Fa...","[Cell Invert Depth, Cell Velocity, Cell Flow B..."
4,nas_profile,11,nas,False,G:\GH\ras-commander\working\hdf_output_options...,"{'write_warmup': True, 'write_time_slices': Fa...","[Cell Invert Depth, Cell Velocity, Cell Flow B..."


In [7]:
compute_records = []

for row in benchmark_plans:
    plan_number = row["plan_number"]
    hdf_path = row["hdf_path"]

    if hdf_path.exists() and not FORCE_RECOMPUTE:
        compute_records.append({
            "label": row["label"],
            "plan_number": plan_number,
            "status": "existing_hdf_reused",
            "elapsed_sec": None,
            "success": True,
            "hdf_path": hdf_path,
        })
        continue

    if not COMPUTE_MISSING_RESULTS:
        compute_records.append({
            "label": row["label"],
            "plan_number": plan_number,
            "status": "missing_hdf_not_computed",
            "elapsed_sec": None,
            "success": False,
            "hdf_path": hdf_path,
        })
        continue

    start = perf_counter()
    result = RasCmdr.compute_plan(
        plan_number,
        ras_object=ras,
        num_cores=NUM_CORES,
        verify=True,
        force_rerun=FORCE_RECOMPUTE,
    )
    elapsed = perf_counter() - start
    compute_records.append({
        "label": row["label"],
        "plan_number": plan_number,
        "status": "computed",
        "elapsed_sec": elapsed,
        "success": bool(result),
        "hdf_path": hdf_path,
    })

refresh_project()
compute_df = pd.DataFrame(compute_records)
compute_df

2026-04-24 13:38:04 - ras_commander.RasCmdr - INFO - Using ras_object with project folder: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D


2026-04-24 13:38:04 - ras_commander.RasUtils - INFO - Using provided plan file path: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07


2026-04-24 13:38:04 - ras_commander.RasUtils - INFO - Successfully updated file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07


2026-04-24 13:38:04 - ras_commander.RasCmdr - INFO - Set number of cores to 4 for plan: 07


2026-04-24 13:38:04 - ras_commander.RasCmdr - INFO - Running HEC-RAS from the Command Line:


...

2026-04-24 14:02:56 - ras_commander.hdf.HdfBase - INFO - Converted WKT to EPSG:2271 from HDF file BaldEagleDamBrk.g09.hdf


2026-04-24 14:02:56 - ras_commander.RasPrj - INFO - ras-commander v0.95.0 | An open-source project of CLB Engineering Corporation (https://clbengineering.com/) | Docs: https://ras-commander.readthedocs.io | GitHub: https://github.com/gpt-cmdr/ras-commander


2026-04-24 14:02:56 - ras_commander.RasPrj - INFO - Project initialized: BaldEagleDamBrk | Folder: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D


2026-04-24 14:02:56 - ras_commander.RasPrj - INFO - Using HEC-RAS executable: C:\Program Files (x86)\HEC\HEC-RAS\7.0\Ras.exe


,label,plan_number,status,elapsed_sec,success,hdf_path
0,baseline_p06_clone,07,computed,291.188715,True,G:\GH\ras-commander\working\hdf_output_options...
1,speed_profile,08,computed,297.617175,True,G:\GH\ras-commander\working\hdf_output_options...
2,balanced_profile,09,computed,297.571243,True,G:\GH\ras-commander\working\hdf_output_options...
3,size_profile,10,computed,308.621530,True,G:\GH\ras-commander\working\hdf_output_options...
4,nas_profile,11,computed,297.411956,True,G:\GH\ras-commander\working\hdf_output_options...


In [8]:
hdf_inventory = []
for row in benchmark_plans:
    hdf_path = row["hdf_path"]
    hdf_inventory.append({
        "label": row["label"],
        "plan_number": row["plan_number"],
        "hdf_exists": hdf_path.exists(),
        "hdf_size_mb": hdf_path.stat().st_size / 1024**2 if hdf_path.exists() else None,
        "hdf_path": hdf_path,
    })

hdf_inventory_df = pd.DataFrame(hdf_inventory)
hdf_inventory_df

,label,plan_number,hdf_exists,hdf_size_mb,hdf_path
0,baseline_p06_clone,07,True,577.267754,G:\GH\ras-commander\working\hdf_output_options...
1,speed_profile,08,True,577.227092,G:\GH\ras-commander\working\hdf_output_options...
2,balanced_profile,09,True,568.167869,G:\GH\ras-commander\working\hdf_output_options...
3,size_profile,10,True,566.233318,G:\GH\ras-commander\working\hdf_output_options...
4,nas_profile,11,True,566.245049,G:\GH\ras-commander\working\hdf_output_options...


In [9]:
missing = hdf_inventory_df[~hdf_inventory_df["hdf_exists"]]
if not missing.empty:
    print("These plans do not have HDF results yet. Set COMPUTE_MISSING_RESULTS = True and run the compute cell.")
    display(missing[["label", "plan_number", "hdf_path"]])

available_hdfs = hdf_inventory_df[hdf_inventory_df["hdf_exists"]].copy()
if available_hdfs.empty:
    raise RuntimeError("No populated HDF files are available for read benchmarking.")

first_hdf = Path(available_hdfs.iloc[0]["hdf_path"])
mesh_names = HdfMesh.get_mesh_area_names(first_hdf)
if not mesh_names:
    raise RuntimeError(f"No mesh areas found in {first_hdf}")

PRIMARY_MESH = mesh_names[0]
print(f"Primary mesh: {PRIMARY_MESH}")
print(mesh_names)

2026-04-24 14:02:56 - ras_commander.hdf.HdfMesh - INFO - Using existing Path object HDF file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07.hdf


2026-04-24 14:02:56 - ras_commander.hdf.HdfMesh - INFO - Final validated file path: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07.hdf


Primary mesh: BaldEagleCr
['BaldEagleCr']


In [10]:
def summarize_output(obj):
    summary = {"result_type": type(obj).__name__}

    if isinstance(obj, dict):
        summary["items"] = len(obj)
        summary["shape"] = str({k: getattr(v, "shape", None) for k, v in obj.items()})
        return summary

    if hasattr(obj, "shape"):
        summary["shape"] = str(obj.shape)

    if hasattr(obj, "__len__"):
        try:
            summary["rows_or_len"] = len(obj)
        except TypeError:
            pass

    if hasattr(obj, "memory_usage"):
        try:
            summary["memory_mb"] = obj.memory_usage(deep=True).sum() / 1024**2
        except Exception:
            pass
    elif hasattr(obj, "nbytes"):
        try:
            summary["memory_mb"] = obj.nbytes / 1024**2
        except Exception:
            pass

    return summary


def extraction_operations(hdf_path: Path, mesh_name: str):
    return [
        ("mesh_area_names", "mesh_geometry", lambda: HdfMesh.get_mesh_area_names(hdf_path)),
        ("mesh_cell_points", "mesh_geometry", lambda: HdfMesh.get_mesh_cell_points(hdf_path)),
        ("mesh_cell_faces", "mesh_geometry", lambda: HdfMesh.get_mesh_cell_faces(hdf_path)),
        ("mesh_cell_polygons", "mesh_geometry", lambda: HdfMesh.get_mesh_cell_polygons(hdf_path)),
        ("mesh_summary_max_ws", "summary_results", lambda: HdfResultsMesh.get_mesh_max_ws(hdf_path)),
        ("mesh_summary_max_face_v", "summary_results", lambda: HdfResultsMesh.get_mesh_max_face_v(hdf_path)),
        ("timeseries_depth", "timeseries_results", lambda: HdfResultsMesh.get_mesh_timeseries(hdf_path, mesh_name, "Depth", truncate=False)),
        ("timeseries_water_surface", "timeseries_results", lambda: HdfResultsMesh.get_mesh_timeseries(hdf_path, mesh_name, "Water Surface", truncate=False)),
        ("max_depth_from_timeseries", "timeseries_results", lambda: HdfResultsMesh.get_mesh_max_depth(hdf_path)),
    ]


def time_operation(label: str, plan_number: str, hdf_path: Path, operation_name: str, category: str, fn, repeat: int):
    gc.collect()
    start = perf_counter()
    try:
        result = fn()
        elapsed = perf_counter() - start
        output_summary = summarize_output(result)
        return {
            "label": label,
            "plan_number": plan_number,
            "operation": operation_name,
            "category": category,
            "repeat": repeat,
            "elapsed_sec": elapsed,
            "success": True,
            "error": None,
            **output_summary,
        }
    except Exception as exc:
        elapsed = perf_counter() - start
        return {
            "label": label,
            "plan_number": plan_number,
            "operation": operation_name,
            "category": category,
            "repeat": repeat,
            "elapsed_sec": elapsed,
            "success": False,
            "error": repr(exc),
            "result_type": None,
        }

In [11]:
read_records = []

for _, hdf_row in available_hdfs.iterrows():
    label = hdf_row["label"]
    plan_number = hdf_row["plan_number"]
    hdf_path = Path(hdf_row["hdf_path"])
    mesh_for_file = HdfMesh.get_mesh_area_names(hdf_path)[0]

    for repeat in range(1, READ_REPETITIONS + 1):
        for operation_name, category, fn in extraction_operations(hdf_path, mesh_for_file):
            read_records.append(
                time_operation(
                    label,
                    plan_number,
                    hdf_path,
                    operation_name,
                    category,
                    fn,
                    repeat,
                )
            )

read_benchmark_df = pd.DataFrame(read_records)
read_benchmark_df

2026-04-24 14:02:56 - ras_commander.hdf.HdfMesh - INFO - Using existing Path object HDF file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07.hdf


2026-04-24 14:02:56 - ras_commander.hdf.HdfMesh - INFO - Final validated file path: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07.hdf


2026-04-24 14:02:56 - ras_commander.hdf.HdfMesh - INFO - Using existing Path object HDF file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07.hdf


2026-04-24 14:02:56 - ras_commander.hdf.HdfMesh - INFO - Final validated file path: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07.hdf


2026-04-24 14:02:56 - ras_commander.hdf.HdfMesh - INFO - Using existing Path object HDF file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07.hdf


...

2026-04-24 14:03:34 - ras_commander.hdf.HdfResultsMesh - INFO - Final validated file path: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p11.hdf


2026-04-24 14:03:34 - ras_commander.hdf.HdfResultsMesh - INFO - Using existing Path object HDF file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p11.hdf


2026-04-24 14:03:34 - ras_commander.hdf.HdfResultsMesh - INFO - Final validated file path: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p11.hdf


2026-04-24 14:03:34 - ras_commander.hdf.HdfResultsMesh - WARNING - Depth dataset not found for mesh 'BaldEagleCr'


,label,plan_number,operation,category,repeat,elapsed_sec,success,error,result_type,rows_or_len,shape,memory_mb
0,baseline_p06_clone,07,mesh_area_names,mesh_geometry,1,0.002444,True,NaN,list,1.0,NaN,NaN
1,baseline_p06_clone,07,mesh_cell_points,mesh_geometry,1,0.129341,True,NaN,GeoDataFrame,19597.0,"(19597, 3)",0.654246
2,baseline_p06_clone,07,mesh_cell_faces,mesh_geometry,1,0.336988,True,NaN,GeoDataFrame,37594.0,"(37594, 3)",1.254961
3,baseline_p06_clone,07,mesh_cell_polygons,mesh_geometry,1,0.925853,True,NaN,GeoDataFrame,18066.0,"(18066, 3)",0.603144
4,baseline_p06_clone,07,mesh_summary_max_ws,summary_results,1,0.172093,True,NaN,GeoDataFrame,19597.0,"(19597, 5)",0.878516
...,...,...,...,...,...,...,...,...,...,...,...,...
130,nas_profile,11,mesh_summary_max_ws,summary_results,3,0.164740,True,NaN,GeoDataFrame,19597.0,"(19597, 5)",0.878516
131,nas_profile,11,mesh_summary_max_face_v,summary_results,3,0.375095,True,NaN,GeoDataFrame,37594.0,"(37594, 5)",1.685190
132,nas_profile,11,timeseries_depth,timeseries_results,3,0.002661,False,ValueError('Failed to get timeseries output: P...,NaN,NaN,NaN,NaN
133,nas_profile,11,timeseries_water_surface,timeseries_results,3,0.228863,True,NaN,DataArray,721.0,"(721, 19597)",53.899525


In [12]:
successful_reads = read_benchmark_df[read_benchmark_df["success"]].copy()

summary_df = (
    successful_reads
    .groupby(["label", "plan_number", "category", "operation"], as_index=False)
    .agg(
        median_elapsed_sec=("elapsed_sec", "median"),
        min_elapsed_sec=("elapsed_sec", "min"),
        max_elapsed_sec=("elapsed_sec", "max"),
        repeats=("elapsed_sec", "count"),
        result_type=("result_type", "first"),
        shape=("shape", "first"),
        rows_or_len=("rows_or_len", "first"),
        memory_mb=("memory_mb", "first"),
    )
    .merge(hdf_inventory_df[["label", "hdf_size_mb"]], on="label", how="left")
    .sort_values(["operation", "median_elapsed_sec"])
)

summary_df

,label,plan_number,category,operation,median_elapsed_sec,min_elapsed_sec,max_elapsed_sec,repeats,result_type,shape,rows_or_len,memory_mb,hdf_size_mb
22,nas_profile,11,timeseries_results,max_depth_from_timeseries,0.004426,0.004355,0.006003,3,GeoDataFrame,"(0, 0)",0.0,0.000126,566.245049
30,size_profile,10,timeseries_results,max_depth_from_timeseries,0.004455,0.004379,0.004622,3,GeoDataFrame,"(0, 0)",0.0,0.000126,566.233318
6,balanced_profile,09,timeseries_results,max_depth_from_timeseries,0.004580,0.004530,0.004604,3,GeoDataFrame,"(0, 0)",0.0,0.000126,568.167869
38,speed_profile,08,timeseries_results,max_depth_from_timeseries,0.004606,0.004563,0.005076,3,GeoDataFrame,"(0, 0)",0.0,0.000126,577.227092
14,baseline_p06_clone,07,timeseries_results,max_depth_from_timeseries,0.005335,0.004565,0.005536,3,GeoDataFrame,"(0, 0)",0.0,0.000126,577.267754
8,baseline_p06_clone,07,mesh_geometry,mesh_area_names,0.002444,0.002382,0.002550,3,list,NaN,1.0,NaN,577.267754
24,size_profile,10,mesh_geometry,mesh_area_names,0.002510,0.002504,0.003206,3,list,NaN,1.0,NaN,566.233318
0,balanced_profile,09,mesh_geometry,mesh_area_names,0.002525,0.002395,0.002826,3,list,NaN,1.0,NaN,568.167869
32,speed_profile,08,mesh_geometry,mesh_area_names,0.002571,0.002463,0.002858,3,list,NaN,1.0,NaN,577.227092
16,nas_profile,11,mesh_geometry,mesh_area_names,0.003406,0.002565,0.005266,3,list,NaN,1.0,NaN,566.245049


In [13]:
pivot = summary_df.pivot_table(
    index=["category", "operation"],
    columns="label",
    values="median_elapsed_sec",
    aggfunc="first",
)

pivot

label                                         balanced_profile  \
category           operation                                     
mesh_geometry      mesh_area_names                    0.002525   
                   mesh_cell_faces                    0.322650   
                   mesh_cell_points                   0.118080   
                   mesh_cell_polygons                 0.868965   
summary_results    mesh_summary_max_face_v            0.360453   
                   mesh_summary_max_ws                0.146122   
timeseries_results max_depth_from_timeseries          0.004580   
                   timeseries_water_surface           0.232250   

label                                         baseline_p06_clone  nas_profile  \
category           operation                                                    
mesh_geometry      mesh_area_names                      0.002444     0.003406   
                   mesh_cell_faces                      0.332694     0.322709   
                   mesh_cell_points                     0.122483     0.124376   
                   mesh_cell_polygons                   0.925853     0.953576   
summary_results    mesh_summary_max_face_v              0.366227     0.375095   
                   mesh_summary_max_ws                  0.150272     0.164740   
timeseries_results max_depth_from_timeseries            0.005335     0.004426   
                   timeseries_water_surface             0.253402     0.228863   

label                                         size_profile  speed_profile  
category           operation                                               
mesh_geometry      mesh_area_names                0.002510       0.002571  
                   mesh_cell_faces                0.318710       0.320000  
                   mesh_cell_points               0.113807       0.121883  
                   mesh_cell_polygons             0.865397       0.871949  
summary_results    mesh_summary_max_face_v        0.357380       0.357957  
                   mesh_summary_max_ws            0.151371       0.146333  
timeseries_results max_depth_from_timeseries      0.004455       0.004606  
                   timeseries_water_surface       0.225420       0.248271

In [14]:
ranking_df = (
    summary_df
    .groupby(["label", "plan_number"], as_index=False)
    .agg(
        total_median_read_sec=("median_elapsed_sec", "sum"),
        hdf_size_mb=("hdf_size_mb", "first"),
    )
)

ranking_df["read_rank"] = ranking_df["total_median_read_sec"].rank(method="min")
ranking_df["size_rank"] = ranking_df["hdf_size_mb"].rank(method="min")
ranking_df["balanced_rank_score"] = ranking_df["read_rank"] + ranking_df["size_rank"]

ranking_df.sort_values(["balanced_rank_score", "total_median_read_sec", "hdf_size_mb"])

,label,plan_number,total_median_read_sec,hdf_size_mb,read_rank,size_rank,balanced_rank_score
3,size_profile,10,2.039050,566.233318,1.0,1.0,2.0
0,balanced_profile,09,2.055625,568.167869,2.0,3.0,5.0
4,speed_profile,08,2.073569,577.227092,3.0,4.0,7.0
2,nas_profile,11,2.177192,566.245049,5.0,2.0,7.0
1,baseline_p06_clone,07,2.158711,577.267754,4.0,5.0,9.0


In [15]:
def inspect_hdf_datasets(hdf_path: Path):
    records = []
    meshes = HdfMesh.get_mesh_area_names(hdf_path)
    with h5py.File(hdf_path, "r") as hdf:
        for mesh_name in meshes:
            candidate_paths = [
                f"Results/Unsteady/Output/Output Blocks/Base Output/Unsteady Time Series/2D Flow Areas/{mesh_name}/Depth",
                f"Results/Unsteady/Output/Output Blocks/Base Output/Unsteady Time Series/2D Flow Areas/{mesh_name}/Water Surface",
                f"Results/Unsteady/Output/Output Blocks/Base Output/Summary Output/2D Flow Areas/{mesh_name}/Maximum Water Surface",
                f"Results/Unsteady/Output/Output Blocks/Base Output/Summary Output/2D Flow Areas/{mesh_name}/Maximum Face Velocity",
            ]
            for dataset_path in candidate_paths:
                if dataset_path not in hdf:
                    continue
                item = hdf[dataset_path]
                if isinstance(item, h5py.Dataset):
                    records.append({
                        "hdf_path": hdf_path,
                        "mesh_name": mesh_name,
                        "dataset": dataset_path.split("/")[-1],
                        "object_type": "dataset",
                        "shape": item.shape,
                        "dtype": str(item.dtype),
                        "chunks": item.chunks,
                        "compression": item.compression,
                        "compression_opts": item.compression_opts,
                    })
                else:
                    records.append({
                        "hdf_path": hdf_path,
                        "mesh_name": mesh_name,
                        "dataset": dataset_path.split("/")[-1],
                        "object_type": "group",
                        "shape": None,
                        "dtype": None,
                        "chunks": None,
                        "compression": None,
                        "compression_opts": None,
                    })
    return records


dataset_records = []
for _, hdf_row in available_hdfs.iterrows():
    for record in inspect_hdf_datasets(Path(hdf_row["hdf_path"])):
        record.update({
            "label": hdf_row["label"],
            "plan_number": hdf_row["plan_number"],
        })
        dataset_records.append(record)

dataset_layout_df = pd.DataFrame(dataset_records)
dataset_layout_df[["label", "plan_number", "mesh_name", "dataset", "shape", "chunks", "compression", "compression_opts"]]

2026-04-24 14:03:35 - ras_commander.hdf.HdfMesh - INFO - Using existing Path object HDF file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07.hdf


2026-04-24 14:03:35 - ras_commander.hdf.HdfMesh - INFO - Final validated file path: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p07.hdf


2026-04-24 14:03:35 - ras_commander.hdf.HdfMesh - INFO - Using existing Path object HDF file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p08.hdf


2026-04-24 14:03:35 - ras_commander.hdf.HdfMesh - INFO - Final validated file path: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p08.hdf


2026-04-24 14:03:35 - ras_commander.hdf.HdfMesh - INFO - Using existing Path object HDF file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p09.hdf


...

2026-04-24 14:03:35 - ras_commander.hdf.HdfMesh - INFO - Using existing Path object HDF file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p10.hdf


2026-04-24 14:03:35 - ras_commander.hdf.HdfMesh - INFO - Final validated file path: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p10.hdf


2026-04-24 14:03:35 - ras_commander.hdf.HdfMesh - INFO - Using existing Path object HDF file: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p11.hdf


2026-04-24 14:03:35 - ras_commander.hdf.HdfMesh - INFO - Final validated file path: G:\GH\ras-commander\working\hdf_output_options_benchmark\BaldEagleCrkMulti2D\BaldEagleDamBrk.p11.hdf


,label,plan_number,mesh_name,dataset,shape,chunks,compression,compression_opts
0,baseline_p06_clone,07,BaldEagleCr,Water Surface,"(721, 19597)","(2, 19597)",gzip,1
1,baseline_p06_clone,07,BaldEagleCr,Maximum Water Surface,"(2, 19597)","(2, 19597)",gzip,1
2,baseline_p06_clone,07,BaldEagleCr,Maximum Face Velocity,"(2, 37594)","(1, 37594)",gzip,1
3,speed_profile,08,BaldEagleCr,Water Surface,"(721, 19597)","(2, 19597)",gzip,1
4,speed_profile,08,BaldEagleCr,Maximum Water Surface,"(2, 19597)","(2, 19597)",gzip,1
5,speed_profile,08,BaldEagleCr,Maximum Face Velocity,"(2, 37594)","(1, 37594)",gzip,1
6,balanced_profile,09,BaldEagleCr,Water Surface,"(721, 19597)","(2, 19597)",gzip,4
7,balanced_profile,09,BaldEagleCr,Maximum Water Surface,"(2, 19597)","(2, 19597)",gzip,4
8,balanced_profile,09,BaldEagleCr,Maximum Face Velocity,"(2, 37594)","(1, 37594)",gzip,4
9,size_profile,10,BaldEagleCr,Water Surface,"(721, 19597)","(2, 19597)",gzip,6


## Interpreting Results

- Use the operation pivot to see whether an HDF setting helps geometry reads, summary reads, or full time-series reads.
- Use the ranking table as a quick read-speed plus file-size score, not as a universal answer.
- Re-run the read benchmark after moving the same populated project to HDD, SSD, and NAS storage. OS cache can dominate short benchmarks, so compare medians over several repetitions and repeat on a quiet machine.
- If additional variables are enabled, only compare against runs where those same variables were enabled; otherwise file size and read timing are not like-for-like.